# Chapter 22: Gradient Boosting from Scratch

## Learning Objectives\n- Understand additive boosting updates\n- Implement residual fitting loop\n- Control bias/variance with key regularization knobs\n- Compare boosting to bagging approaches

## Prerequisites\n- Python basics and functions\n- Numpy basics for deep learning chapters\n- Understanding of earlier chapters (0-19)\n

# Chapter 22: Gradient Boosting from Scratch

## Why this chapter matters
Gradient boosting is one of the highest-impact tabular modeling families in industry.
Understanding residual fitting and additive modeling helps you reason about XGBoost/LightGBM/CatBoost.

## Core idea
Build the model in stages:
\[
F_M(x) = F_{M-1}(x) + \eta h_M(x)
\]
where each weak learner \(h_M\) is trained on pseudo-residuals.

## Step-by-step (regression)
1. Initialize with constant prediction: mean(y).
2. Compute residuals: `r = y - y_pred`.
3. Fit a weak learner on `(X, r)`.
4. Update predictions: `y_pred += lr * learner_pred`.
5. Repeat for `n_estimators`.

## Key regularization knobs
- learning rate (`eta`)
- tree depth / stump complexity
- row/feature subsampling
- early stopping on validation split

## Industry guidance
1. Start with shallow trees (depth 3-8 for libraries, stumps/depth-1 for pure Python practice).
2. Prefer lower learning rate + more trees.
3. Monitor calibration and class imbalance.
4. Use SHAP/feature attribution for business explanation.

## Assignment
1. Implement boosting for regression from scratch.
2. Extend to binary classification with log-loss gradients.
3. Compare with random forest on same split.



In [ ]:
"""Tiny gradient boosting regressor with decision stumps.
Pure Python educational implementation.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import List, Sequence


@dataclass
class Stump:
    feature_idx: int
    threshold: float
    left_value: float
    right_value: float

    def predict_one(self, x: Sequence[float]) -> float:
        return self.left_value if x[self.feature_idx] <= self.threshold else self.right_value


def mean(values: Sequence[float]) -> float:
    return sum(values) / max(1, len(values))


def fit_stump(X: List[List[float]], residuals: List[float]) -> Stump:
    n_features = len(X[0])
    best = None
    best_sse = float("inf")

    for j in range(n_features):
        thresholds = sorted({row[j] for row in X})
        for t in thresholds:
            left = [residuals[i] for i, row in enumerate(X) if row[j] <= t]
            right = [residuals[i] for i, row in enumerate(X) if row[j] > t]
            if not left or not right:
                continue

            lv = mean(left)
            rv = mean(right)
            sse = 0.0
            for i, row in enumerate(X):
                pred = lv if row[j] <= t else rv
                diff = residuals[i] - pred
                sse += diff * diff

            if sse < best_sse:
                best_sse = sse
                best = Stump(j, t, lv, rv)

    if best is None:
        # Fallback when no split found
        avg = mean(residuals)
        best = Stump(0, X[0][0], avg, avg)
    return best


class GradientBoostingRegressor:
    def __init__(self, n_estimators: int = 50, learning_rate: float = 0.1) -> None:
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.init_value = 0.0
        self.models: List[Stump] = []

    def fit(self, X: List[List[float]], y: List[float]) -> None:
        self.init_value = mean(y)
        y_pred = [self.init_value for _ in y]

        for _ in range(self.n_estimators):
            residuals = [yt - yp for yt, yp in zip(y, y_pred)]
            stump = fit_stump(X, residuals)
            self.models.append(stump)

            for i, row in enumerate(X):
                y_pred[i] += self.learning_rate * stump.predict_one(row)

    def predict(self, X: List[List[float]]) -> List[float]:
        preds = [self.init_value for _ in X]
        for stump in self.models:
            for i, row in enumerate(X):
                preds[i] += self.learning_rate * stump.predict_one(row)
        return preds


def mse(y_true: List[float], y_pred: List[float]) -> float:
    return sum((a - b) ** 2 for a, b in zip(y_true, y_pred)) / len(y_true)


if __name__ == "__main__":
    X = [[1.0], [2.0], [3.0], [4.0], [5.0], [6.0]]
    y = [1.1, 1.9, 3.2, 3.9, 5.1, 6.2]

    model = GradientBoostingRegressor(n_estimators=20, learning_rate=0.2)
    model.fit(X, y)
    pred = model.predict(X)
    print("MSE:", mse(y, pred))
    print("Pred:", [round(v, 3) for v in pred])


## Checkpoint\n\n1. You can derive residual update steps.\n2. You can explain learning-rate vs number-of-trees tradeoff.\n3. You can identify overfitting signs in boosting.

## Guided Exercise\nTrain with several `n_estimators` and `learning_rate` combinations and compare MSE.

In [ ]:
# TODO (Student Exercise)
X = [[1.0],[2.0],[3.0],[4.0],[5.0],[6.0]]
y = [1.1, 1.9, 3.2, 3.9, 5.1, 6.2]
for n, lr in [(5,0.3),(20,0.2),(80,0.05)]:
    m = GradientBoostingRegressor(n_estimators=n, learning_rate=lr)
    m.fit(X,y)
    p = m.predict(X)
    print(n, lr, mse(y,p))

## Quick Quiz\n\n**Q1.** What does each weak learner fit in boosting?\n\n**Answer:** Pseudo-residuals/negative gradients of current model.\n\n**Q2.** Why use shrinkage (small learning rate)?\n\n**Answer:** It regularizes updates and improves generalization.\n\n**Q3.** One early-stopping signal?\n\n**Answer:** Validation error starts rising while train error keeps dropping.\n

## Homework\nExtend this implementation to binary classification with logistic loss gradients.